# PDBFrame: protein structures as a DataFrame

A `PDBFrame` is an atom table, with one row per atom. Every method returns a new `PDBFrame`, a number or a table; nothing is changed in place. For 3D viewing, see `pdbframe_display.ipynb`.

*The 3D views aren't saved in this file, to keep it small. Run the cells to see them.*

In [1]:
from pdbframe import PDBFrame

ref = PDBFrame.read('data/3tfk_rcsb.pdb')            # experimental structure
model = PDBFrame.read('data/3tfk_af2_renamed.pdb')   # AlphaFold2 model, chains named differently
ref

,chain,residue_number,insertion_code,observed_index,residue_name,aa,atom_name,element,altloc,x,y,z,occupancy,bfactor,hetero
0,A,2,,1,PRO,P,N,N,,13.040,-46.388000,-29.325001,1.0,98.78,False
1,A,2,,1,PRO,P,CA,C,,14.095,-46.644001,-28.337999,1.0,98.78,False
2,A,2,,1,PRO,P,C,C,,15.091,-45.493000,-28.271999,1.0,98.78,False
3,A,2,,1,PRO,P,O,O,,15.853,-45.285000,-29.215000,1.0,98.78,False
4,A,2,,1,PRO,P,CB,C,,14.779,-47.902000,-28.881001,1.0,115.89,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4967,E,257,,239,ASP,D,O,O,,-6.054,-55.521999,60.702999,1.0,112.87,False
4968,E,257,,239,ASP,D,CB,C,,-4.803,-53.179001,60.122002,1.0,117.41,False
4969,E,257,,239,ASP,D,CG,C,,-4.006,-51.904999,59.931999,1.0,117.41,False
4970,E,257,,239,ASP,D,OD1,O,,-3.134,-51.618000,60.778000,1.0,117.41,False


Structures can also be loaded straight from the RCSB PDB by ID (this needs internet access). The default is the deposited structure as mmCIF; `assembly=1` loads the biological assembly, and `save='data/'` keeps the downloaded file. See the end of this notebook for FASTA sequences.

In [2]:
PDBFrame.from_rcsb('3tfk', metadata=True)

,chain,residue_number,insertion_code,observed_index,residue_name,aa,atom_name,element,altloc,x,y,z,occupancy,bfactor,hetero
0,A,2,,1,PRO,P,N,N,,13.040,-46.388000,-29.325001,1.0,98.78,False
1,A,2,,1,PRO,P,CA,C,,14.095,-46.644001,-28.337999,1.0,98.78,False
2,A,2,,1,PRO,P,C,C,,15.091,-45.493000,-28.271999,1.0,98.78,False
3,A,2,,1,PRO,P,O,O,,15.853,-45.285000,-29.215000,1.0,98.78,False
4,A,2,,1,PRO,P,CB,C,,14.779,-47.902000,-28.881001,1.0,115.89,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4967,D,241,,239,ASP,D,O,O,,-6.054,-55.521999,60.702999,1.0,112.87,False
4968,D,241,,239,ASP,D,CB,C,,-4.803,-53.179001,60.122002,1.0,117.41,False
4969,D,241,,239,ASP,D,CG,C,,-4.006,-51.904999,59.931999,1.0,117.41,False
4970,D,241,,239,ASP,D,OD1,O,,-3.134,-51.618000,60.778000,1.0,117.41,False


## The columns

| column | meaning |
|---|---|
| `chain` | chain ID |
| `residue_number` | residue number as written in the file (can start anywhere, skip numbers or be negative) |
| `insertion_code` | letter after the residue number in the file, e.g. the `A` in `100A`; empty for most residues |
| `observed_index` | 1-based position of the residue among its chain's residues with atoms *in this frame*, recomputed after every selection (`pf.residues` also has `full_index`, its position in the deposited sequence, for mmCIF files) |
| `residue_name` | three-letter residue name, e.g. `ALA` |
| `aa` | one-letter code, `X` for non-standard residues |
| `atom_name` | atom name, e.g. `CA` |
| `element` | chemical element |
| `altloc` | alternate-location label (only one location is kept) |
| `x`, `y`, `z` | coordinates (Å) |
| `occupancy` | occupancy |
| `bfactor` | B-factor; in predicted models this column holds the pLDDT |
| `hetero` | True for HETATM records (ligands, ions), which are only read with `read(..., hetero=True)` |

`residue_number` + `insertion_code` is the file's own label for a residue. `observed_index` is just its position in the chain, so two files with different numbering still have comparable positions.

In [3]:
ref.select(chains='A', residues=(1, 4), atoms='ca')   # chain A starts at residue_number 2, observed_index 1

,chain,residue_number,insertion_code,observed_index,residue_name,aa,atom_name,element,altloc,x,y,z,occupancy,bfactor,hetero
0,A,2,,1,PRO,P,CA,C,,14.095000,-46.644001,-28.337999,1.0,98.78,False
1,A,3,,2,HIS,H,CA,C,,16.000000,-43.640999,-26.966999,1.0,115.39,False
2,A,4,,3,SER,S,CA,C,,18.559999,-43.046001,-24.205999,1.0,64.52,False


## What's inside

In [4]:
ref.sequences

{'A': 'PHSMRYYETATSRRGLGEPRYTSVGYVDDKEFVRFDSDAENPRYEPQVPWMEQEGPEYWERITQVAKGQEQWFRVNLRTLLGYYNQSAGGTHTLQRMYGCDVGSDGRLLRGYEQFAYDGCDYIALNEDLRTWTAADMAAQITRRKWEQAGAAEYYRAYLEGECVEWLHRYLKNG',
 'C': 'QLSDVPMDL',
 'D': 'AQSVTQPDARVTVSEGASLQLRCKYSYSATPYLFWYVQYPRQGLQMLLKYYSGDPVVQGVNGFEAEFSKSDSSFHLRKASVHWSDSAVYFCAVSAKGTGSKLSFGKGAKLTVSPNIQNPDPAVYQLRDSKSSDKSVCLFTDFDSQTNVSQSKDSDVYITDKCVLDMRSMDFKSNSAVAWSNKSDFACANAFNNSIIPEDTF',
 'E': 'AVTQSPRNKVTVTGGNVTLSCRQTNSHNYMYWYRQDTGHGLRLIHYSYGAGNLQIGDVPDGYKATRTTQEDFFLLLELASPSQTSLYFCASSDAPGQLYFGEGSKLTVLEDLKNVFPPEVAVFEPSEAEISHTQKATLVCLATGFYPDHVELSWWVNGKEVHSGVCTDPQPLKEQPALNDSRYALSSRLRVSATFWQNPRNHFRCQVQFYGLSENDEWTQDRAKPVTQIVSAEAWGRAD'}

In [5]:
ref.residues.head()   # one row per residue

,chain,observed_index,residue_number,insertion_code,residue_name,aa,atoms,observed,full_index
0,A,1,2,,PRO,P,7,True,<NA>
1,A,2,3,,HIS,H,10,True,<NA>
2,A,3,4,,SER,S,6,True,<NA>
3,A,4,5,,MET,M,8,True,<NA>
4,A,5,6,,ARG,R,11,True,<NA>


## Select
Filter by chain, residue numbers (an inclusive range of `residue_number`), named atom sets (`'heavy'`, `'backbone'`, `'ca'`) or atom names. Filters combine.

In [6]:
ref.select(chains='A', residues=(50, 60), atoms='backbone')

,chain,residue_number,insertion_code,observed_index,residue_name,aa,atom_name,element,altloc,x,y,z,occupancy,bfactor,hetero
0,A,50,,1,PRO,P,N,N,,22.472000,-26.971001,-30.825001,1.0,99.78,False
1,A,50,,1,PRO,P,CA,C,,21.589001,-25.910000,-31.327999,1.0,99.78,False
2,A,50,,1,PRO,P,C,C,,20.275000,-25.764000,-30.556000,1.0,99.78,False
3,A,50,,1,PRO,P,O,O,,20.063999,-24.739000,-29.910999,1.0,99.78,False
4,A,51,,2,TRP,W,N,N,,19.410000,-26.770000,-30.617001,1.0,103.21,False
5,A,51,,2,TRP,W,CA,C,,18.077999,-26.662001,-30.030001,1.0,103.21,False
6,A,51,,2,TRP,W,C,C,,18.091000,-26.224001,-28.566000,1.0,103.21,False
7,A,51,,2,TRP,W,O,O,,17.302000,-25.372000,-28.155001,1.0,103.21,False
8,A,52,,3,MET,M,N,N,,19.003000,-26.797001,-27.791000,1.0,102.11,False
9,A,52,,3,MET,M,CA,C,,19.004999,-26.622000,-26.344999,1.0,102.11,False


In [7]:
peptide = ref.select(chains='C')
pocket = ref.near(peptide, cutoff=5)   # whole residues within 5 Å of the peptide
pocket.residues.groupby('chain').size()

chain
A    32
C     9
D     7
E     7
dtype: int64

In [8]:
ref.contacts(cutoff=4).head()   # residue pairs between chains

,chain_1,residue_number_1,insertion_code_1,residue_name_1,chain_2,residue_number_2,insertion_code_2,residue_name_2,distance
0,A,45,,TYR,C,2,,LEU,3.830314
1,A,59,,TYR,C,1,,GLN,3.953389
2,A,62,,ARG,C,1,,GLN,2.814938
3,A,63,,ILE,C,2,,LEU,3.988969
4,A,66,,VAL,C,1,,GLN,3.422459


## Compare a model with the reference
The model's chain IDs don't match the reference's, as often happens with prediction tools:

In [9]:
ref.chains, model.chains

(['A', 'C', 'D', 'E'], ['H', 'P', 'X', 'Y'])

Nothing needs renaming, because the two structures are paired automatically:
- **Chains** are matched by sequence: every model chain is aligned with every reference chain using `SeqFrame`, and chains are paired one-to-one to maximize identical residues.
- **Residues** are matched within each chain pair by the same alignment, so numbering can differ and residues can be missing. Here, chain H in the model is numbered one higher than chain A in the reference.
- **Atoms** are matched by name.

In [10]:
model.map_chains(ref)   # {model chain: reference chain}

{'H': 'A', 'P': 'C', 'X': 'D', 'Y': 'E'}

This is the alignment behind one of those pairs, using a chain's `SeqFrame`:

In [11]:
model.seqframe('P').align(ref.seqframe('C')).show(one_line=True)

P     1 QLSDVPMDL 9
        |||||||||
C     1 QLSDVPMDL 9



All scores are under `.metrics`. `summary` gives the main ones, plus the chain pairing it used:

In [12]:
model.metrics.summary(ref)

rmsd                                        20.049377
tm_score                                     0.643133
lddt                                         0.756971
plddt                                        89.86297
chain_map    {'H': 'A', 'P': 'C', 'X': 'D', 'Y': 'E'}
Name: 3tfk_af2_renamed, dtype: object

In [13]:
print('RMSD (CA) ', model.metrics.rmsd(ref))
print('TM-score  ', model.metrics.tm_score(ref))
print('lDDT      ', model.metrics.lddt(ref))
print('pLDDT     ', model.metrics.plddt())

RMSD (CA)  20.049377132456424
TM-score   0.6431327822989504
lDDT       0.7569714306889225
pLDDT      89.8629695024077


Restrict a comparison to some of the model's chains with `chains=`. They're still matched to the reference automatically:

In [14]:
{c: round(model.metrics.rmsd(ref, chains=[c]), 2) for c in model.chains}

{'H': 0.77, 'P': 0.77, 'X': 15.95, 'Y': 14.04}

DockQ scores one interface. Name the two sides with the **reference's** chain IDs:

In [15]:
model.metrics.dockq(ref, 'A', 'C')

{'dockq': 0.9141900763490195,
 'fnat': 0.9298245614035088,
 'irmsd': 0.6581725223432656,
 'lrmsd': 1.3834859347063593,
 'fnonnat': 0.22058823529411764,
 'native_contacts': 57,
 'model_contacts': 68}

In [16]:
model.metrics.lddt(ref, per_residue=True).head()   # per residue of the reference

,chain,observed_index,residue_number,insertion_code,residue_name,aa,atoms,observed,full_index,lddt
0,A,1,2,,PRO,P,7,True,<NA>,0.850102
1,A,2,3,,HIS,H,10,True,<NA>,0.838882
2,A,3,4,,SER,S,6,True,<NA>,0.843476
3,A,4,5,,MET,M,8,True,<NA>,0.795780
4,A,5,6,,ARG,R,11,True,<NA>,0.856807


To give the model the reference's chain IDs, pass the reference to `rename_chains`:

In [17]:
model.rename_chains(ref).chains

['A', 'C', 'D', 'E']

To control the pairing yourself, pass `chains={model_chain: reference_chain}`. `chains='same'` pairs by chain ID only, which finds nothing here:

In [18]:
model.metrics.rmsd(ref, chains={'H': 'A'})

0.7705863780614562

## Superpose and save
`superpose` moves the whole structure onto the reference, by the best fit of the chosen atoms and chains. Here the fit uses chain H only (the MHC, chain A in the reference). `write()` saves to PDB, mmCIF or CSV, depending on the file extension.

The test model is already superposed on the reference, so we first move it away, the way a raw prediction would arrive.

**Colors: grey = reference, orange = model.**

In [19]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler('xyz', [40, 20, 60], degrees=True).as_matrix()
raw = model.transform(R, [30, 0, 0])
print('chain H RMSD before:', raw.metrics.rmsd(ref, chains=['H'], superpose=False))
print('whole complex before:', raw.metrics.rmsd(ref, superpose=False))
raw.py3Dmol(ref, colors=['orange', 'lightgrey'])      # before: orange model, grey reference

chain H RMSD before: 58.25862037519936
whole complex before: 75.57058117876913


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
fitted = raw.superpose(ref, chains=['H'])          # fit on model chain H (reference chain A)
print('chain H RMSD after:', fitted.metrics.rmsd(ref, chains=['H'], superpose=False))
print('whole complex after:', fitted.metrics.rmsd(ref, superpose=False), '(the TCR chains sit differently)')
fitted.py3Dmol(ref, colors=['orange', 'lightgrey'])   # after: orange model, grey reference

chain H RMSD after: 0.7705863780614564
whole complex after: 24.233000209534307 (the TCR chains sit differently)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [21]:
print(fitted.to_string()[:240])   # or fitted.write('model_on_ref.pdb' / '.cif' / '.csv')

ATOM      1  N   PRO H   3      14.559 -46.576 -30.120  1.00 89.62           N  
ATOM      2  CA  PRO H   3      14.958 -46.450 -28.728  1.00 89.62           C  
ATOM      3  C   PRO H   3      15.813 -45.188 -28.575  1.00 89.62           C


## From the RCSB, and sequences without coordinates
`PDBFrame.from_rcsb('3tfk')` downloads an entry (mmCIF by default; `assembly=1` for the biological assembly, `save='data/'` to keep the file). Note that RCSB's chain IDs (A, B, C, D) differ from the processed file above.

With `fmt='fasta'` you get the entry's full sequences instead: a `PDBFrame` without coordinates. It has one row per residue, numbered from 1, with chains named as in the structure. The same frame comes from any `.fasta` file through `PDBFrame.read()`, or from `PDBFrame.from_sequences({...})`. Selecting, `sequences`, `residues` and matching work as usual; anything needing coordinates raises a clear error.

In [22]:
entry = PDBFrame.from_rcsb('3tfk')                 # needs internet
seqs = PDBFrame.from_rcsb('3tfk', fmt='fasta')
entry, seqs

(PDBFrame '3TFK': 4 chains (A, B, C, D), 623 residues, 4972 atoms
      chain  residue_number insertion_code  observed_index residue_name aa  \
 0        A               2                              1          PRO  P   
 1        A               2                              1          PRO  P   
 2        A               2                              1          PRO  P   
 3        A               2                              1          PRO  P   
 4        A               2                              1          PRO  P   
 ...    ...             ...            ...             ...          ... ..   
 4967     D             241                            239          ASP  D   
 4968     D             241                            239          ASP  D   
 4969     D             241                            239          ASP  D   
 4970     D             241                            239          ASP  D   
 4971     D             241                            239          ASP  D  

`PDBFrame.rcsb_info` gives the entries' metadata (title, release date, method, resolution, number of polymer entities), one row per entry - handy for filtering a set of structures by date or resolution.

In [23]:
PDBFrame.rcsb_info(['3tfk', '1ao7'])

,pdb_id,title,release_date,method,resolution,n_entities
0,3TFK,42F3-p4B10/H2-Ld,2011-12-07 00:00:00+00:00,X-ray,2.753,4
1,1AO7,"COMPLEX BETWEEN HUMAN T-CELL RECEPTOR, VIRAL P...",1997-09-17 00:00:00+00:00,X-ray,2.600,5


`from_rcsb(..., metadata=True)` fetches the same row with the structure and keeps it in `pf.metadata`; selections keep it too.

In [24]:
entry = PDBFrame.from_rcsb('3tfk', metadata=True)
entry.metadata

{'pdb_id': '3TFK',
 'title': '42F3-p4B10/H2-Ld',
 'release_date': Timestamp('2011-12-07 00:00:00+0000', tz='UTC'),
 'method': 'X-ray',
 'resolution': 2.753,
 'n_entities': 4}

In [25]:
seqs.select(chains='A', residues=(1, 10)).sequences

{'A': 'MGPHSMRYYE'}

The FASTA holds the full deposited construct, so matching it to the structure shows which residues were never modeled:

In [26]:
mine, _ = seqs.match(entry)
modeled = set(zip(mine.df['chain'], mine.df['residue_number']))
res = seqs.residues
res[[(c, n) not in modeled for c, n in zip(res['chain'], res['residue_number'])]]

,chain,observed_index,residue_number,insertion_code,residue_name,aa,atoms,observed,full_index
0,A,1,1,,MET,M,1,True,<NA>
1,A,2,2,,GLY,G,1,True,<NA>
176,A,177,177,,ASN,N,1,True,<NA>
177,A,178,178,,ALA,A,1,True,<NA>
178,A,179,179,,THR,T,1,True,<NA>
179,A,180,180,,LEU,L,1,True,<NA>
189,C,1,1,,GLY,G,1,True,<NA>
190,C,2,2,,SER,S,1,True,<NA>
191,C,3,3,,HIS,H,1,True,<NA>
192,C,4,4,,MET,M,1,True,<NA>
